# 01 - Universe Filtering

The Guardrail Alpha agent only trades a curated BSC universe. This
notebook loads `configs/eligible_assets.bsc.json`, shows the full
20-token universe, and breaks it down by category.

**Offline-safe:** if the config is missing the notebook prints a
hint instead of raising.


In [ ]:
# --- Guardrail Alpha notebook bootstrap (offline-safe) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Return the first ancestor that contains python-lab/guardrail_lab."""
    for candidate in [start, *start.parents]:
        if (candidate / "python-lab" / "guardrail_lab").is_dir():
            return candidate
    return start


# In a notebook __file__ is undefined, so anchor on the current working dir.
REPO_ROOT = _find_repo_root(Path.cwd())
LAB_PATH = REPO_ROOT / "python-lab"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))


def _first_existing(*candidates: Path):
    """Return the first path that exists, else None."""
    for path in candidates:
        if path.exists():
            return path
    return None


DATA_DIR = REPO_ROOT / "data"
CONFIG_DIR = REPO_ROOT / "configs"

# Prefer a real run; fall back to the seeded demo artifacts if present.
DB_PATH = _first_existing(
    DATA_DIR / "guardrail_alpha.db",
    DATA_DIR / "demo_guardrail_alpha.db",
)
RUN_REPORT_PATH = _first_existing(
    DATA_DIR / "run_report.json",
    DATA_DIR / "demo_run_report.json",
)
EXPERIMENTS_DIR = DATA_DIR / "experiments"
BACKTESTS_DIR = DATA_DIR / "backtests"
ELIGIBLE_ASSETS_PATH = CONFIG_DIR / "eligible_assets.bsc.json"
ASSET_CATEGORIES_PATH = CONFIG_DIR / "asset_categories.json"

NO_DATA_HINT = (
    "No data found under data/. Run the agent (paper/live) or seed a demo run "
    "first, e.g.  python3 -m guardrail_lab.seed  (from python-lab/), then "
    "re-run this notebook. The notebook is offline-safe and will not raise."
)

print("repo root :", REPO_ROOT)
print("event log :", DB_PATH if DB_PATH else "(none yet)")
print("run report:", RUN_REPORT_PATH if RUN_REPORT_PATH else "(none yet)")


In [ ]:
def render_table(rows, columns, title=None):
    """Print an aligned text table. rows: list[dict]; columns: list[(key,label)]."""
    if title:
        print(title)
        print("=" * len(title))
    if not rows:
        print("(no rows)")
        return
    labels = [label for _, label in columns]
    keys = [key for key, _ in columns]
    cells = [[("" if r.get(k) is None else str(r.get(k))) for k in keys] for r in rows]
    widths = [
        max(len(labels[i]), *(len(row[i]) for row in cells)) for i in range(len(keys))
    ]
    header = "  ".join(labels[i].ljust(widths[i]) for i in range(len(keys)))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(keys))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(keys))))


## Load the eligible-asset universe


In [ ]:
import json

universe = []
if ELIGIBLE_ASSETS_PATH.exists():
    try:
        universe = json.loads(ELIGIBLE_ASSETS_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as exc:
        print("Could not parse eligible_assets.bsc.json:", exc)
        universe = []

if not universe:
    print(NO_DATA_HINT)
else:
    enabled = [a for a in universe if a.get("enabled")]
    print(f"Universe size: {len(universe)} tokens "
          f"({len(enabled)} enabled).")


## Full universe table


In [ ]:
if universe:
    rows = [
        {
            "symbol": a.get("symbol"),
            "category": a.get("category"),
            "cmc_id": a.get("cmc_id"),
            "enabled": a.get("enabled"),
            "min_liq_usd": a.get("min_liquidity_usd"),
            "min_vol24h_usd": a.get("min_volume_24h_usd"),
        }
        for a in sorted(universe, key=lambda x: (x.get("category", ""), x.get("symbol", "")))
    ]
    render_table(
        rows,
        [
            ("symbol", "SYMBOL"),
            ("category", "CATEGORY"),
            ("cmc_id", "CMC_ID"),
            ("enabled", "ENABLED"),
            ("min_liq_usd", "MIN_LIQ_USD"),
            ("min_vol24h_usd", "MIN_VOL24H_USD"),
        ],
        title="Eligible BSC universe",
    )
else:
    print(NO_DATA_HINT)


## Category breakdown


In [ ]:
if universe:
    by_cat = {}
    for a in universe:
        cat = a.get("category", "unknown")
        by_cat.setdefault(cat, []).append(a.get("symbol"))

    cat_rows = [
        {"category": cat, "count": len(syms), "symbols": ", ".join(sorted(s for s in syms if s))}
        for cat, syms in sorted(by_cat.items(), key=lambda kv: (-len(kv[1]), kv[0]))
    ]
    render_table(
        cat_rows,
        [("category", "CATEGORY"), ("count", "COUNT"), ("symbols", "SYMBOLS")],
        title="Universe by category",
    )
else:
    print(NO_DATA_HINT)


## Notes

* `min_liquidity_usd` / `min_volume_24h_usd` are the pre-trade
  eligibility gates the agent enforces before an asset can be
  scored or traded.
* Stable assets (e.g. USDT/USDC) are the settlement legs; the agent
  rotates risk capital across the non-stable categories.
